# First-N Pruning — ФИНАЛЬНЫЙ НОУТБУК (полный пайплайн)

**Вход**: целая (dense) nnUNet-модель fold_0 из input. **Выход**: pruned INT8 IR + все Dice + замеры скорости.

Пайплайн (все фазы с чекпоинтами и логированием):
1. ENV + DEPS + загрузка модели и данных
2. **CONFIG** — все настройки с комментариями (глубина резки, recovery, INT8)
3. Скан глубины (опционально): Dice на каждом уровне резки → выбор safe-глубин
4. Поэтапная резка с recovery KD (iterative — одна группа за шаг, гейты после каждого шага)
5. Финальная валидация pruned-модели
6. Экспорт ONNX → IR FP32
7. INT8 NNCF (subprocess) + argmax-гейт на прод-подобных патчах
8. Тайминги FP32/INT8 + итоговый отчёт JSON

Опирается на проверенные прогоны 2026-10-17..24: iterative [32,32,64,128,192,160,160] →
recovered 0.8778 (fold-val), INT8 0.8375 (10 прод-кейсов), малые структуры паритет.


In [ ]:
# ============ CELL 1: ENV ============
import os, multiprocessing as mp, time
import sys
for _s in (sys.stdout, sys.stderr):
    try:
        _s.reconfigure(line_buffering=True)   # чистый TextIOWrapper
    except AttributeError:
        try: _s.line_buffering = True          # ipykernel OutStream
        except Exception: pass
mp.set_start_method('spawn', force=True)
os.environ['nnUNet_raw'] = '/kaggle/working/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/kaggle/working/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/kaggle/working/nnUNet_results'
os.environ.setdefault('nnUNet_n_proc_DA', '4')
os.makedirs('/kaggle/working/nnUNet_results', exist_ok=True)

KERNEL_T0 = time.time()
SMOKE = False
EXPORT_ONLY = False
SESSION_BUDGET_H = 11.0   # жёсткий стоп по времени сессии — страховка GPU-лимита 30ч/нед
print('env OK | SMOKE =', SMOKE, '| session budget =', SESSION_BUDGET_H, 'ч')


In [ ]:
# ============ CELL 2: DEPS ============
import subprocess, sys, importlib.metadata as im
def _pip(*a): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], check=True)
try:
    v = im.version('nnunetv2')
except Exception:
    _pip('nnunetv2'); v = im.version('nnunetv2')
print('nnunetv2', v)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'onnx', 'onnxruntime', 'openvino', 'nncf', 'onnxscript'], check=False)
import torch, glob, json, copy, gc, random, shutil
import numpy as np
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| GPUs:', torch.cuda.device_count())

In [ ]:
# ============ CELL 3: PATHS + BASE + CONFIG ============
from batchgenerators.utilities.file_and_folder_operations import join, load_json, save_json
from dynamic_network_architectures.architectures.unet import ResidualEncoderUNet
import importlib

def find_input_root():
    hits = sorted(glob.glob('/kaggle/input/**/nnUNetTrainerDistill__*/fold_0/checkpoint_final.pth',
                            recursive=True))
    assert hits, 'Дистиллят не найден в /kaggle/input'
    ds_dir = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    return os.path.dirname(os.path.dirname(ds_dir))

INP = find_input_root()
print('INP =', INP)

DATASET_ID = 1
FOLD = 0
TRAINER_FOLDER = 'nnUNetTrainerDistill__nnUNetResEncUNetMPlans__3d_fullres'
PLANS_NAME = 'nnUNetResEncUNetMPlans'
CONFIG = '3d_fullres'
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

pre_dir = glob.glob(f'{INP}/nnUNet_preprocessed/Dataset{DATASET_ID:03d}_*')[0]
DATASET_NAME = os.path.basename(pre_dir)
plans_base = load_json(join(pre_dir, f'{PLANS_NAME}.json'))
dataset_json = load_json(join(pre_dir, 'dataset.json'))
cfg = plans_base['configurations'][CONFIG]
kw = dict(cfg['architecture']['arch_kwargs'])
FEATS_BASE = list(kw['features_per_stage'])

_modality = dataset_json.get('channel_names', dataset_json.get('modality', {}))
IN_CH = len(_modality) if isinstance(_modality, dict) else int(_modality or 1)
IN_CH = max(IN_CH, 1)
NUM_CLASSES = len(dataset_json['labels'])
print(f'каналов: {IN_CH} | классов (с фоном): {NUM_CLASSES} | FEATS_BASE: {FEATS_BASE}')

def _resolve_class(full_path):
    module_path, class_name = full_path.rsplit('.', 1)
    return getattr(importlib.import_module(module_path), class_name)

ds_work_root = '/kaggle/working/nnUNet_preprocessed'
ds_work = join(ds_work_root, DATASET_NAME)
OUT = '/kaggle/working/phaseB_outputs'

def build_net(feat_override=None, dense=False):
    if dense:
        feats = list(kw['features_per_stage']); kw_use = kw
    else:
        plans_now = load_json(join(ds_work, f'{PLANS_NAME}.json'))
        cfg_now = plans_now['configurations'][CONFIG]
        kw_use = cfg_now['architecture']['arch_kwargs']
        feats = feat_override if feat_override is not None else list(kw_use['features_per_stage'])
    if len(feats) == 7:
        feats = feats[1:]
    assert len(feats) == 6, f'features_per_stage: ожидалось 6, получено {len(feats)}'
    return ResidualEncoderUNet(
        input_channels=IN_CH, num_classes=NUM_CLASSES,
        n_stages=kw_use['n_stages'], features_per_stage=list(feats),
        conv_op=_resolve_class(kw_use['conv_op']),
        kernel_sizes=kw_use['kernel_sizes'], strides=kw_use['strides'],
        n_blocks_per_stage=kw_use['n_blocks_per_stage'],
        n_conv_per_stage_decoder=kw_use['n_conv_per_stage_decoder'],
        conv_bias=kw_use['conv_bias'],
        norm_op=_resolve_class(kw_use['norm_op']), norm_op_kwargs=kw_use['norm_op_kwargs'],
        dropout_op=None, dropout_op_kwargs=None,
        nonlin=_resolve_class(kw_use['nonlin']), nonlin_kwargs=kw_use['nonlin_kwargs'])

dist_hits = sorted(glob.glob('/kaggle/input/**/nnUNetTrainerDistill__*/fold_' + str(FOLD) + '/checkpoint_final.pth', recursive=True))
assert dist_hits, 'Дистиллят-чекпоинт не найден'
ck_path = dist_hits[0]
print('checkpoint:', ck_path)
ckpt = torch.load(ck_path, map_location='cpu', weights_only=False)

pd_hits = sorted(glob.glob(f'{INP}/**/{PLANS_NAME}.json', recursive=True))
assert pd_hits, f'{PLANS_NAME}.json не найден'
plans_dense = load_json(pd_hits[0])
cfg_d = plans_dense['configurations'][CONFIG]
kw_d = dict(cfg_d['architecture']['arch_kwargs'])
assert list(kw_d['features_per_stage']) == FEATS_BASE, 'plans дистиллята не плотные'

BASE = build_net(dense=True).cuda()
missing, unexpected = BASE.load_state_dict(ckpt['network_weights'], strict=False)
assert not missing and not unexpected, f'Рассинхрон: {missing[:3]} {unexpected[:3]}'
BASE.eval()
print(f'BASE (dense): {sum(p.numel() for p in BASE.parameters())/1e6:.1f}M')

if EXPORT_ONLY:
    rec_hits = sorted(glob.glob('/kaggle/input/**/recovered/checkpoint_final.pth', recursive=True))
    assert rec_hits, 'recovered/checkpoint_final.pth не найден в /kaggle/input'
    REC_CK_PATH = rec_hits[0]
    rec_ckpt = torch.load(REC_CK_PATH, map_location='cpu', weights_only=False)
    rec_sd = rec_ckpt['network_weights']

# ---- КОНФИГ РЕЗКИ: enc3 через Taylor-DC+CE ----
CUT_RATIOS = {'enc3': 0.15, 'enc4': 0.20, 'enc5': 0.20}
CRITERIA   = {'enc3': 'taylor_dcce', 'enc4': 'l2', 'enc5': 'l2'}
MIN_CHANNELS = {'stem': 20, 'enc0': 20, 'enc1': 28, 'enc2': 40,
                'enc3': 80, 'enc4': 96, 'enc5': 96}
DICE_BUDGET       = 0.02
RECOVERY_EPOCHS = 60
RECOVERY_LR = 1e-4
RECOVERY_WD = 3e-5
RECOVERY_MOMENTUM = 0.9
BUDGET_TRAIN_H = 9.5
print('CUT_RATIOS:', CUT_RATIOS, '| CRITERIA:', CRITERIA)

In [ ]:
# ================================================================
# КОНФИГУРАЦИЯ ПАЙПЛАЙНА — единственное место настроек
# ================================================================

# ---------- РЕЖИМ ЗАПУСКА ----------
RUN_DEPTH_SCAN = True    # ФАЗА A: скан глубины — валидация Dice на каждом уровне резки
                         # (шаг DEPTH_STEP каналов от dense к минимуму) для выбора
                         # safe-глубин. ~8 мин/уровень на 2×T4. False = пропустить
                         # (если глубины уже известны — задать вручную ниже).
RUN_ITERATIVE  = True    # ФАЗА B: поэтапная резка + recovery KD (основной путь).
                         # False = только one-shot резка без recovery (не рекомендуется:
                         # комбо-провал 0.6935 — см. cookbook).
RUN_EXPORT_INT8 = True   # ФАЗА C: ONNX → IR FP32 → INT8 NNCF → гейт → тайминги.
RUN_TIMING = True        # Замер скорости инференса FP32 vs INT8 (CPU, per-patch).

# ---------- ГЛУБИНА РЕЗКИ (главная опция) ----------
# Какие стадии резать и до скольких каналов (кратно 32 — тайлы INT8/AVX).
# ПЛОТНЫЕ ширины стадий нашей ResEnc-M: stem/enc0=32, enc1=64, enc2=128, enc3=256, enc4=enc5=320.
# Проверенные безопасные глубины (гейт Dice ≥0.86 на fold-val, сканы 2026-10-17..24):
#   stem/enc0  — резать НЕЛЬЗЯ (−9% → Dice 0.77 и ниже)
#   enc1       — почти нельзя (−12.5% → 0.8558 — ниже гейта)
#   enc2       — до 120 из 128 (0.8671; глубже 112 → 0.8560 — провал)
#   enc3       — до 192 из 256 (0.8620; с recovery возвращается к 0.8776)
#   enc4/enc5  — до 32 из 320 (0.8768 — почти бесплатно; в iterative брали 160)
# PRUNE_PLAN = {'имя_стадии': целевое_число_каналов, ...} — только стадии, которые режем.
# СТАДИИ РЕЖУТСЯ ПО ОДНОЙ ЗА ШАГ (iterative — резать всё сразу = комбо-провал!),
# порядок = порядок ключей словаря (рекомендация: от глубоких к мелким).
PRUNE_PLAN = {
    'enc3': 192,    # −25% ширины: one-shot 0.8620 → recovered 0.8776 (шаг 1, проверено)
    'enc4': 160,    # −50%: поверх recovered шага 1 → 0.8767 → recovered 0.8778 (шаг 2, проверено)
    'enc5': 160,    # enc4 и enc5 — связанная пара, всегда режутся одинаково
}

# ---------- СКАН ГЛУБИНЫ (ФАЗА A) ----------
DEPTH_STAGES = ['enc3', 'enc4']  # какие стадии сканировать (по одной, независимо)
DEPTH_STEP = 32                  # шаг скана в каналах (32 = кратно INT8-тайлам)
DEPTH_MIN = {'enc3': 128, 'enc4': 96, 'enc5': 96}  # нижняя граница скана по стадиям
DEPTH_GATE = 0.86                # гейт Dice для уровня: ниже = стоп скана этой стадии

# ---------- ИТЕРАТИВНАЯ РЕЗКА + RECOVERY (ФАЗА B) ----------
RECOVERY_EPOCHS = 60      # эпох recovery после каждого шага резки (проверено: 60 достаточно,
                          # вернуло +0.0157 на шаге 1 и +0.0011 на шаге 2)
RECOVERY_LR = 1e-4        # LR SGD. ВАЖНО: не дефолт nnUNet (0.01) — он разрушает редкие классы
RECOVERY_WD = 3e-5        # weight decay
RECOVERY_MOMENTUM = 0.9   # momentum. ВАЖНО: не 0.99 (дефолт nnUNet) — эффективный шаг ×100 рвёт Dice
TEACHER_FOLD0 = True      # учитель = плотная модель fold_0 (проверено: 356 с/эпоха, хватает)
TEACHER_ENSEMBLE = True   # попробовать ансамбль 5 фолдов из датасета int8model (если найдётся —
                          # средний softmax, лучше таргеты; авто-фоллбек на fold_0)
DICE_GATE = 0.86          # обязательный гейт после КАЖДОГО шага (ниже — стоп пайплайна)
DICE_TARGET = 0.875       # цель для перехода к следующему шагу (мягкий ориентир)

# ---------- КВАНТИЗАЦИЯ (ФАЗА C) ----------
CALIB_PATCHES = 100       # патчей калибровки NNCF из ПРОД-подобных данных (resample+crop).
                          # КРИТИЧНО: 100 патчей с периферией; fg-центрированные train-патчи
                          # (даже 300 шт.) дают нерабочий INT8 (кризис 0.0004 — cookbook §3)
CALIB_CASES = 12          # из скольких кейсов брать патчи
CALIB_CT_NORM = True      # калибровка в продовой CTNormalization (clip p0.5..p99.5 → scale)
                          # — НЕ per-patch z-норма (искажает диапазоны активаций)
ARGMAX_GATE = 0.999       # гейт согласия argmax INT8 vs FP32 (проверяется на своих патчах)
INT8_PRESET = 'MIXED'     # пресет NNCF (MIXED — безопасный выбор для сегментации)

# ---------- БЮДЖЕТ СЕССИИ ----------
SESSION_BUDGET_H = 11.0   # мягкий бюджет-стоп (12 ч — жёсткий лимит Kaggle; запас на
                          # финальную валидацию и экспорт обязателен)

# ---------- СЛУЖЕБНОЕ (не менять без необходимости) ----------
RES_DIR_GLOBAL = '/kaggle/working/pipeline_out'   # папка всех артефактов пайплайна
RES_DIR = RES_DIR_GLOBAL                            # алиас (валидатор/тайминги из старых ячеек)
os.makedirs(RES_DIR_GLOBAL, exist_ok=True)
import sys as _sys
for _s in (_sys.stdout, _sys.stderr):   # живой лог (ipykernel-совместимо)
    try:
        _s.reconfigure(line_buffering=True)
    except AttributeError:
        try: _s.line_buffering = True
        except Exception: pass
print('CONFIG OK | план резки:', PRUNE_PLAN)


In [ ]:
# ============ CELL 4: РЕЗКА (План C v12) + ТЕСТЫ ============
import copy
import torch.nn as _nn

ENC_STAGES = ['enc5', 'enc4', 'enc3', 'enc2', 'enc1', 'enc0', 'stem']
CHAIN      = ['stem', 'enc0', 'enc1', 'enc2', 'enc3', 'enc4', 'enc5']

def get_blocks(container):
    for attr in ('blocks', 'convs'):
        if hasattr(container, attr):
            return list(getattr(container, attr))
    raise TypeError(f'{type(container).__name__}: нет списка блоков')

def _stage_container(net, st):
    return net.encoder.stem if st == 'stem' else net.encoder.stages[int(st[3:])]

def _blocks_of(net, st):
    return list(_stage_container(net, st).convs) if st == 'stem' \
           else get_blocks(_stage_container(net, st))

def _units_of(block):
    out = []
    for tag in ('conv1', 'conv2', 'conv'):
        b = getattr(block, tag, None)
        if b is None:
            continue
        conv = getattr(b, 'conv', b)
        if isinstance(conv, _nn.modules.conv._ConvNd):
            out.append((tag, conv, getattr(b, 'norm', None)))
    sk = getattr(block, 'skip', None)
    if sk is not None and isinstance(sk, _nn.Module):
        for sub in sk:
            conv = getattr(sub, 'conv', sub)
            if isinstance(conv, _nn.modules.conv._ConvNd):
                out.append(('skip', conv, getattr(sub, 'norm', None)))
    assert out, f'{type(block).__name__}: нет свёрток'
    return out

if 'DECODER_CAT_SKIP_FIRST' not in globals():
    def _detect_cat_order():
        import inspect, importlib, pkgutil
        import dynamic_network_architectures as dna
        dec_cls = None
        for mod_name in ('dynamic_network_architectures.building_blocks.simple_decoders',
                         'dynamic_network_architectures.building_blocks.unet',
                         'dynamic_network_architectures.architectures.unet'):
            try:
                m = importlib.import_module(mod_name)
                if hasattr(m, 'UNetDecoder'):
                    dec_cls = m.UNetDecoder
                    print('UNetDecoder:', mod_name)
                    break
            except ModuleNotFoundError:
                continue
        if dec_cls is None:
            for finder, name, ispkg in pkgutil.walk_packages(dna.__path__, dna.__name__ + '.'):
                try:
                    m = importlib.import_module(name)
                except Exception:
                    continue
                if hasattr(m, 'UNetDecoder'):
                    dec_cls = m.UNetDecoder
                    print('UNetDecoder:', name)
                    break
        assert dec_cls is not None, 'UNetDecoder не найден'
        src = inspect.getsource(dec_cls.forward)
        for line in src.splitlines():
            s = line.strip()
            if '.cat(' in s or 'torch.cat' in s:
                print(' cat:', s)
                inner = s[s.find('('):]
                first_is_x = inner.strip('( [').startswith('x')
                return not first_is_x
        raise RuntimeError('torch.cat не найден')
    DECODER_CAT_SKIP_FIRST = _detect_cat_order()
    print('DECODER_CAT_SKIP_FIRST =', DECODER_CAT_SKIP_FIRST)

def _slice_out(conv, keep, norm=None):
    k = keep.to(conv.weight.device)
    conv.weight.data = conv.weight.data.index_select(0, k).contiguous()
    if conv.bias is not None:
        conv.bias.data = conv.bias.data.index_select(0, k).contiguous()
    conv.out_channels = len(k)
    if norm is not None and getattr(norm, 'weight', None) is not None:
        norm.weight.data = norm.weight.data.index_select(0, k).contiguous()
        norm.bias.data = norm.bias.data.index_select(0, k).contiguous()
    if norm is not None and getattr(norm, 'num_features', None) is not None:
        norm.num_features = len(k)

def _slice_in(conv, keep):
    k = keep.to(conv.weight.device)
    conv.weight.data = conv.weight.data.index_select(1, k).contiguous()
    conv.in_channels = len(k)

def _force_norm_sync(block, n_out, keep=None):
    for name, m in block.named_modules():
        if isinstance(m, (_nn.InstanceNorm1d, _nn.InstanceNorm2d, _nn.InstanceNorm3d,
                          _nn.BatchNorm1d, _nn.BatchNorm2d, _nn.BatchNorm3d)):
            if m.weight is not None and m.weight.shape[0] != n_out:
                if keep is not None:
                    k = keep.to(m.weight.device)
                    m.weight.data = m.weight.data.index_select(0, k).contiguous()
                    if m.bias is not None:
                        m.bias.data = m.bias.data.index_select(0, k).contiguous()
                m.num_features = n_out

def _normalize_skips(enc, CHAIN):
    changed = 0
    for st in CHAIN:
        cont = enc.stem if st == 'stem' else enc.stages[int(st[3:])]
        blocks = list(cont.convs) if st == 'stem' else get_blocks(cont)
        for b in blocks:
            sk = getattr(b, 'skip', None)
            if sk is None or not isinstance(sk, _nn.Module):
                continue
            conv1_stride = None
            c1 = getattr(b, 'conv1', None) or getattr(b, 'conv', None)
            if c1 is not None:
                cc = getattr(c1, 'conv', c1)
                conv1_stride = getattr(cc, 'stride', (1, 1, 1))
            conv1x1 = None
            for sub in sk:
                cv = getattr(sub, 'conv', sub)
                if isinstance(cv, _nn.modules.conv._ConvNd):
                    conv1x1 = cv
                    break
            if conv1x1 is not None and conv1x1.in_channels == conv1x1.out_channels:
                avg = None
                for sub in sk:
                    if isinstance(sub, _nn.AvgPool3d):
                        avg = sub
                        break
                if avg is not None:
                    b.skip = _nn.Sequential(_nn.AvgPool3d(
                        kernel_size=avg.kernel_size, stride=avg.stride, padding=avg.padding))
                elif conv1_stride is not None and all(s == 1 for s in conv1_stride):
                    b.skip = (lambda x: x)
                else:
                    b.skip = _nn.Sequential(_nn.AvgPool3d(
                        kernel_size=conv1_stride, stride=conv1_stride, padding=0))
                changed += 1
    if changed:
        print(f'  нормализация skip: удалено избыточных skip-conv: {changed}')
    return changed

def _chain_counts(net):
    return {st: _units_of(_blocks_of(net, st)[0])[0][1].out_channels for st in CHAIN}

def _keep_top(net, st, C_i, n_keep):
    if n_keep >= C_i:
        return None
    score = torch.zeros(C_i)
    for b in _blocks_of(net, st):
        for tag, conv, norm in _units_of(b):
            if conv.weight.shape[0] == C_i:
                score += conv.weight.data.reshape(conv.weight.shape[0], -1)\
                            .norm(dim=1).detach().cpu()
    return torch.topk(score, n_keep).indices.sort().values

def _keep_idx(net, st, C_i, n_keep, taylor_dcce=None):
    """Диспетчер критериев: 'taylor_dcce' (enc3), 'l2' (enc4/enc5), 'first'."""
    if n_keep >= C_i:
        return None
    if CRITERIA.get(st, 'l2') == 'first':
        return torch.arange(n_keep)
    if CRITERIA.get(st, 'l2') == 'taylor_dcce' and taylor_dcce is not None:
        agg = None
        for (s_, bi, tag), sc in taylor_dcce.items():
            if s_ == st and sc.shape[0] == C_i:
                agg = sc if agg is None else agg + sc
        if agg is not None:
            return torch.topk(agg, n_keep).indices.sort().values
        print(f'  ⚠️ taylor_dcce пуст для {st} — fallback на l2')
    return _keep_top(net, st, C_i, n_keep)

def _skip_has_conv(block):
    sk = getattr(block, 'skip', None)
    if sk is None or not isinstance(sk, _nn.Module):
        return False
    for sub in sk:
        conv = getattr(sub, 'conv', sub)
        if isinstance(conv, _nn.modules.conv._ConvNd):
            return True
    return False

def _constrained_pairs(net):
    pairs = []
    for a, b in [(CHAIN[i], CHAIN[i+1]) for i in range(len(CHAIN)-1)]:
        if not _skip_has_conv(_blocks_of(net, b)[0]):
            pairs.append((a, b))
    print('Пары с обязательным равенством каналов:', pairs)
    return pairs

def _prune_encoder_obj(enc, fK, CHAIN):
    prev_keep = None
    for st in CHAIN:
        keep = fK[st]
        cont = enc.stem if st == 'stem' else enc.stages[int(st[3:])]
        blocks = list(cont.convs) if st == 'stem' else get_blocks(cont)
        for bi, b in enumerate(blocks):
            for tag, conv, norm in _units_of(b):
                _slice_out(conv, keep, norm)
                if st == 'stem':
                    if bi > 0:
                        _slice_in(conv, keep)
                else:
                    if tag == 'conv2' or bi > 0:
                        _slice_in(conv, keep)
                    elif tag in ('conv1', 'skip') and prev_keep is not None:
                        _slice_in(conv, prev_keep)
            _force_norm_sync(b, len(keep), keep)
        prev_keep = keep

def _sync_duplicate_encoder(net, fK, CHAIN):
    enc_dup = getattr(net.decoder, 'encoder', None)
    if enc_dup is None:
        return
    named_src = dict(net.encoder.named_modules())
    src_params = dict(net.encoder.named_parameters())
    for name, md_ in enc_dup.named_modules():
        ms = named_src.get(name)
        if isinstance(ms, _nn.modules.conv._ConvNd) and isinstance(md_, _nn.modules.conv._ConvNd):
            md_.in_channels = ms.in_channels
            md_.out_channels = ms.out_channels
            md_.weight = _nn.Parameter(ms.weight.data.clone())
            if ms.bias is not None:
                md_.bias = _nn.Parameter(ms.bias.data.clone())
    for name, p_src in src_params.items():
        parts = name.split('.')
        mod = enc_dup
        try:
            for pth in parts[:-1]:
                mod = getattr(mod, pth)
            last = parts[-1]
            cur_p = getattr(mod, last, None)
            if cur_p is not None and cur_p.shape != p_src.shape:
                setattr(mod, last, _nn.Parameter(p_src.data.clone()))
        except AttributeError:
            continue
    for st in CHAIN:
        keep = fK[st]
        cont = enc_dup.stem if st == 'stem' else enc_dup.stages[int(st[3:])]
        blocks = list(cont.convs) if st == 'stem' else get_blocks(cont)
        for b in blocks:
            _force_norm_sync(b, len(keep), keep)
    print('  decoder.encoder: синхронизирован (conv+norm+все параметры)')

def apply_stage_ratios(net, ratios, taylor_dcce=None):
    net = net.cpu()
    C = _chain_counts(net)
    n = {}
    for st in CHAIN:
        r = ratios.get(st, 0)
        n[st] = C[st] if r <= 0 else min(max(int(round(C[st]*(1-r))),
                                             MIN_CHANNELS.get(st, 4)), C[st])
    for a, b in _constrained_pairs(net):
        m = min(n[a], n[b])
        if (n[a], n[b]) != (m, m):
            print(f'  ограничение {a}↔{b}: {n[a]}/{n[b]} → {m}/{m}')
        n[a] = n[b] = m
    fK = {}
    for st in CHAIN:
        kk = _keep_idx(net, st, C[st], n[st], taylor_dcce=taylor_dcce)
        fK[st] = kk if kk is not None else torch.arange(C[st])
    _prune_encoder_obj(net.encoder, fK, CHAIN)
    _normalize_skips(net.encoder, CHAIN)
    _sync_duplicate_encoder(net, fK, CHAIN)
    _normalize_skips(net.decoder.encoder, CHAIN)
    for k in range(len(net.decoder.transpconvs)):
        Ks = fK[CHAIN[5 - k]]
        tc = net.decoder.transpconvs[k]
        if k == 0:
            Kb = fK['enc5']
            tc.weight.data = tc.weight.data.index_select(0, Kb).contiguous()
            tc.in_channels = len(Kb)
        else:
            Kprev = fK[CHAIN[5 - (k - 1)]]
            tc.weight.data = tc.weight.data.index_select(0, Kprev).contiguous()
            tc.in_channels = len(Kprev)
        tc.weight.data = tc.weight.data.index_select(1, Ks).contiguous()
        if tc.bias is not None:
            tc.bias.data = tc.bias.data.index_select(0, Ks).contiguous()
        tc.out_channels = len(Ks)
        unit = net.decoder.stages[k].convs[0]
        conv = getattr(unit, 'conv', unit)
        norm = getattr(unit, 'norm', None)
        w = conv.weight.data
        half = w.shape[1] // 2
        if half == len(Ks):
            conv.in_channels = 2 * len(Ks)
        else:
            up_part   = w[:, :half].index_select(1, Ks.to(w.device)).contiguous()
            skip_part = w[:, half:].index_select(1, Ks.to(w.device)).contiguous()
            parts = [skip_part, up_part] if DECODER_CAT_SKIP_FIRST else [up_part, skip_part]
            w_new = torch.cat(parts, dim=1).contiguous()
            conv.weight.data = w_new
            conv.in_channels = w_new.shape[1]
        _slice_out(conv, Ks, norm)
        _force_norm_sync(unit, len(Ks), Ks)
        _slice_in(net.decoder.seg_layers[k], Ks)
    Cn = {st: _units_of(_blocks_of(net, st)[0])[0][1].out_channels for st in CHAIN}
    for st in CHAIN:
        flag = '⚠️' if Cn[st] < MIN_CHANNELS.get(st, 0) else '✓'
        print(f'  {st}: {C[st]} → {Cn[st]} {flag}')
    enc_dup = getattr(net.decoder, 'encoder', None)
    if enc_dup is not None:
        cn2 = _units_of(get_blocks(enc_dup.stages[0])[0])[0][1].out_channels
        ok = '✓ согласован' if cn2 == Cn['enc0'] else '⚠️ рассинхрон'
        print(f'  decoder.encoder: enc0={cn2} {ok}')

# ---- Форматы каналов (из CELL 6 вашего ноутбука) ----
def fix_monotonic(fp6, strides):
    fp6 = list(fp6)
    for i in range(1, len(fp6)):
        m = int(strides[i][0] * strides[i][1] * strides[i][2])
        fp6[i] = max(int(min(fp6[i], fp6[i-1] * m)), 8)
    return fp6

def plans6_to_feats7(fp6):
    return [fp6[0]] + list(fp6)

def feats7_to_plans6(feats7):
    assert feats7[0] == feats7[1], (
        f'stem({feats7[0]}) != enc0({feats7[1]}) — ограничитель пары не сработал')
    return list(feats7[1:])

def encoder_feats_from_net(net):
    feats = [net.encoder.stem.convs[0].conv.out_channels]
    for st in net.encoder.stages:
        last_block = get_blocks(st)[-1]
        last_conv = None
        for tag in ('conv2', 'conv1', 'conv'):
            b = getattr(last_block, tag, None)
            if b is not None:
                last_conv = getattr(b, 'conv', b)
                break
        assert last_conv is not None
        feats.append(last_conv.out_channels)
    return feats

def sync_plans(feats7):
    fp = fix_monotonic(feats7_to_plans6(feats7), kw['strides'])
    p = load_json(join(ds_work, f'{PLANS_NAME}.json'))
    a = p['configurations'][CONFIG]['architecture']['arch_kwargs']
    a['features_per_stage'] = fp
    p['continue_training'] = False
    save_json(p, join(ds_work, f'{PLANS_NAME}.json'))
    return fp

In [ ]:
# ============ CELL 5: РАБОЧАЯ КОПИЯ + ВАЛИДАТОР + BASELINE ============
INP_PRE = f'{INP}/nnUNet_preprocessed'
ds_src = pre_dir
data_id = cfg.get('data_identifier', 'nnUNetPlans_3d_fullres')
real_data = join(ds_src, data_id)
if not os.path.isdir(real_data):
    cand = [d for d in glob.glob(join(ds_src, '*_3d_fullres')) if os.path.isdir(d)]
    assert cand, f'нет папки данных в {ds_src}'
    real_data = cand[0]
print('данные:', real_data)

if not os.path.exists(ds_work):
    os.makedirs(ds_work)
    for fn in ('dataset.json', 'dataset_fingerprint.json', 'splits_final.json',
               f'{PLANS_NAME}.json'):
        shutil.copy(join(ds_src, fn), join(ds_work, fn))
    if os.path.isdir(join(ds_src, 'gt_segmentations')):
        shutil.copytree(join(ds_src, 'gt_segmentations'), join(ds_work, 'gt_segmentations'))
if not os.path.exists(join(ds_work, data_id)):
    os.symlink(real_data, join(ds_work, data_id))
os.environ['nnUNet_preprocessed'] = ds_work_root
print('рабочая копия:', ds_work)

BASELINE_ARCHIVE = f'{OUT}/validations/BASELINE_summary.json'

def make_stub_checkpoint(network_weights, net_ref=None):
    if net_ref is None:
        net_ref = BASE
    dummy_opt = torch.optim.SGD(net_ref.parameters(), lr=1e-3)
    return {
        'network_weights': network_weights,
        'optimizer_state': dummy_opt.state_dict(),
        'grad_scaler_state': None,
        'logging': ckpt.get('logging'),
        '_best_ema': ckpt.get('_best_ema'),
        'current_epoch': ckpt.get('current_epoch', 0),
        'init_args': ckpt.get('init_args'),
        'trainer_name': 'nnUNetTrainer',
        'inference_allowed_mirroring_axes': None,
    }

def dice_from_summary(p):
    s = json.load(open(p))
    m = s.get('foreground_mean') or s.get('mean') or {}
    for k in ('DICE', 'dice', 'Dice'):
        if k in m:
            return float(m[k]), s
    raise KeyError(f'нет dice в {p}: {list(m.keys())[:5]}')

def full_validate(net_or_sd, tag, net_ref=None):
    dst = (f'/kaggle/working/nnUNet_results/{DATASET_NAME}/'
           f'nnUNetTrainer__{PLANS_NAME}__{CONFIG}/fold_{FOLD}')
    os.makedirs(dst, exist_ok=True)
    torch.save(make_stub_checkpoint(net_or_sd, net_ref=net_ref),
               join(dst, 'checkpoint_final.pth'))
    if os.path.exists(join(dst, 'validation')):
        shutil.rmtree(join(dst, 'validation'))
    r = subprocess.run(['nnUNetv2_train', str(DATASET_ID), CONFIG, str(FOLD),
                        '-tr', 'nnUNetTrainer', '-p', PLANS_NAME, '--val'],
                       capture_output=True, text=True, timeout=7200)
    s_path = join(dst, 'validation/summary.json')
    assert os.path.exists(s_path), (
        f'валидация не дала summary.json\n{r.stdout[-1200:]}\n{r.stderr[-1200:]}')
    d, s = dice_from_summary(s_path)
    os.makedirs(f'{OUT}/validations', exist_ok=True)
    shutil.copy(s_path, f'{OUT}/validations/{tag}_summary.json')
    print(f'[{tag}] mean Dice = {d:.4f}')
    return d, s

if EXPORT_ONLY:
    BASELINE_DICE = 0.8800
    sync_plans(plans6_to_feats7(FEATS_BASE))
elif SMOKE:
    BASELINE_DICE = 0.8800
    sync_plans(plans6_to_feats7(FEATS_BASE))
    print('SMOKE: baseline взят из факт-листа, валидация пропущена')
elif os.path.exists(BASELINE_ARCHIVE):
    BASELINE_DICE, _ = dice_from_summary(BASELINE_ARCHIVE)
    sync_plans(plans6_to_feats7(FEATS_BASE))
    print('Baseline из архива:', BASELINE_DICE)
else:
    BASELINE_DICE = 0.8800
    sync_plans(plans6_to_feats7(FEATS_BASE))
    print('Baseline 0.8800 из факт-листа v9 (тот же input; валидация baseline пропущена — экономия GPU)')
FLOOR = BASELINE_DICE - DICE_BUDGET
print(f'FLOOR = {FLOOR:.4f}')

In [ ]:
# ============ ПАРАЛЛЕЛЬНЫЙ ВАЛИДАТОР (2×T4, kaggle-two-gpu-parallelism) ============
# Очередь вариантов -> 2 subprocess-воркера (CUDA_VISIBLE_DEVICES=0/1).
# Правила: разные nnUNet_results на воркера (конфликт checkpoint_final.pth),
# per-variant json-кэш (повтор не пересчитывает), метки [W0]/[W1] в логе,
# nnUNet_n_proc_DA снижен (4 vCPU на двоих).
import threading, queue as _q

W_RESULTS = ['/kaggle/working/nnUNet_results_W0', '/kaggle/working/nnUNet_results_W1']
for p in W_RESULTS:
    os.makedirs(p, exist_ok=True)
os.environ.setdefault('nnUNet_n_proc_DA', '2')

_val_q = _q.Queue()
_val_res = {}
_val_lock = threading.Lock()

def _worker_loop(wid):
    res_dir = W_RESULTS[wid]
    env = dict(os.environ)
    env['CUDA_VISIBLE_DEVICES'] = str(wid)
    env['nnUNet_results'] = res_dir
    env['OMP_NUM_THREADS'] = '2'
    while True:
        task = _val_q.get()
        if task is None:
            # sentinel: НЕ выходим из цикла (v5-урок: мёртвые потоки ломают
            # второй вызов validate_many — очередь никто не читает) — просто ждём дальше
            continue
        tag, ck_payload = task
        cache = f'{RES_DIR}/val_cache_{tag}.json'
        if os.path.exists(cache):
            d = json.load(open(cache))['dice']
            with _val_lock:
                _val_res[tag] = (d, json.load(open(cache)).get('summary'))
            print(f'[W{wid}] {tag}: кэш, Dice={d:.4f}', flush=True)
            continue
        dst = join(res_dir, DATASET_NAME, f'nnUNetTrainer__{PLANS_NAME}__{CONFIG}', f'fold_{FOLD}')
        os.makedirs(dst, exist_ok=True)
        torch.save(ck_payload, join(dst, 'checkpoint_final.pth'))
        if os.path.exists(join(dst, 'validation')):
            shutil.rmtree(join(dst, 'validation'))
        t0 = time.time()
        r = subprocess.run(['nnUNetv2_train', str(DATASET_ID), CONFIG, str(FOLD),
                            '-tr', 'nnUNetTrainer', '-p', PLANS_NAME, '--val'],
                           capture_output=True, text=True, timeout=14400, env=env)
        s_path = join(dst, 'validation/summary.json')
        if not os.path.exists(s_path):
            print(f'[W{wid}] {tag}: ВАЛИДАЦИЯ УПАЛА\n{r.stdout[-800:]}\n{r.stderr[-800:]}', flush=True)
            with _val_lock:
                _val_res[tag] = (None, None)
            continue
        d, s = dice_from_summary(s_path)
        os.makedirs(f'{RES_DIR}', exist_ok=True)
        shutil.copy(s_path, f'{RES_DIR}/{tag}_summary.json')
        json.dump({'dice': d, 'summary': s, 'worker': wid,
                   'elapsed_s': time.time() - t0}, open(cache, 'w'), indent=1)
        with _val_lock:
            _val_res[tag] = (d, s)
        print(f'[W{wid}] {tag}: mean Dice = {d:.4f} ({(time.time()-t0)/60:.1f} мин)', flush=True)

_val_threads = {}
def _alive(wid):
    th = _val_threads.get(wid)
    return th is not None and th.is_alive()

def _ensure_worker(wid):
    if not _alive(wid):
        th = threading.Thread(target=_worker_loop, args=(wid,), daemon=True)
        th.start()
        _val_threads[wid] = th
        print(f'воркер W{wid} запущен', flush=True)

def start_validators():
    for wid in (0, 1):
        _ensure_worker(wid)
    print('параллельные валидаторы готовы (W0 cuda:0, W1 cuda:1)')

def validate_many(jobs, timeout_s=14400):
    """jobs: список (tag, checkpoint_payload). Возвращает {tag: dice} (только удачные)."""
    start_validators()   # идемпотентно: перезапускает умершие воркеры
    with _val_lock:
        for tag, _ in jobs:
            _val_res.pop(tag, None)   # сброс старых результатов тех же тегов
    for j in jobs:
        _val_q.put(j)
    pending = {tag for tag, _ in jobs}
    t0 = time.time()
    while pending and (time.time() - t0) < timeout_s:
        with _val_lock:
            done = {tag for tag in pending if tag in _val_res}
        pending -= done
        if pending:
            time.sleep(5)
    if pending:
        print(f'validate_many: таймаут ожидания {pending}', flush=True)
    out = {}
    failed = [tag for tag, _ in jobs if _val_res.get(tag, (None, None))[0] is None]
    if failed:
        print(f'ВАЛИДАЦИИ НЕ ДАЛИ РЕЗУЛЬТАТА (частичный сбой): {failed}', flush=True)
    for tag, _ in jobs:
        d, _s = _val_res.get(tag, (None, None))
        if d is not None:
            out[tag] = d
    return out

def stub_for(sd, net_ref):
    return make_stub_checkpoint(sd, net_ref=net_ref)

start_validators()


In [ ]:
# ============ ФАЗА A: СКАН ГЛУБИНЫ (Dice на каждом уровне) ============
# Для каждой стадии из DEPTH_STAGES: one-shot first-N резка от dense к DEPTH_MIN с шагом
# DEPTH_STEP, полная валидация каждого уровня (параллельные воркеры W0/W1), стоп при Dice < DEPTH_GATE.
# Результат: depth_scan.json — кривая Dice по каналам каждой стадии.
import numpy as np

depth_scan = {}
if RUN_DEPTH_SCAN:
    for stage in DEPTH_STAGES:
        base_ch = {'enc3': 256, 'enc4': 320, 'enc5': 320, 'enc2': 128, 'enc1': 64}[stage]
        lo = DEPTH_MIN.get(stage, base_ch // 2)
        levels = list(range(base_ch - DEPTH_STEP, lo - 1, -DEPTH_STEP))
        results = []
        for n_keep in levels:
            if (time.time() - KERNEL_T0) > SESSION_BUDGET_H * 3600 * 0.4:
                print(f'[budget] скан остановлен на {stage}={n_keep}')
                break
            # готовим уровень: dense-plans → резка одной стадии → plans под резку
            sync_plans(plans6_to_feats7(FEATS_BASE))
            net_s = copy.deepcopy(BASE)
            apply_stage_ratios(net_s, {stage: 1 - n_keep / base_ch})
            feats_s = encoder_feats_from_net(net_s)
            sync_plans(feats_s)
            sd_s = {k: v.cpu() for k, v in net_s.state_dict().items()}
            tag = f'SCAN_{stage}_{n_keep}'
            d = validate_many([(tag, stub_for(sd_s, net_s))]).get(tag)
            net_s = net_s.cpu(); del net_s
            gc.collect(); torch.cuda.empty_cache()
            results.append({'stage': stage, 'channels': n_keep, 'dice': d})
            print(f'{stage}={n_keep}: Dice={d:.4f} {"OK" if d >= DEPTH_GATE else "СТОП (ниже гейта)"}')
            if d is None or d < DEPTH_GATE:
                break
        depth_scan[stage] = results
        # вернуть плотные plans для следующей стадии
        sync_plans(plans6_to_feats7(FEATS_BASE))
    json.dump(depth_scan, open(f'{RES_DIR_GLOBAL}/depth_scan.json', 'w'), indent=1)
    print('сохранено: depth_scan.json')
    # безопасные глубины = последний прошедший гейт уровень каждой стадии
    safe = {st: max((r['channels'] for r in rs if r['dice'] and r['dice'] >= DEPTH_GATE),
                    default=None) for st, rs in depth_scan.items()}
    print('SAFE-глубины по скану:', safe, '— при желании обновите PRUNE_PLAN и перезапустите фазы B/C')
else:
    print('ФАЗА A пропущена (RUN_DEPTH_SCAN=False) — используем PRUNE_PLAN как задан')


In [ ]:
# ============ ФАЗА B: ПОЭТАПНАЯ РЕЗКА + RECOVERY KD (iterative) ============
# Порядок стадий = порядок ключей PRUNE_PLAN. После каждого шага:
# one-shot валидация → recovery KD (учитель fold_0/ансамбль) → валидация recovered → гейт DICE_GATE.
import torch.nn.functional as _F
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer

TEACHER_DEVICE = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'

def _load_teacher(sd):
    t = copy.deepcopy(BASE).to(TEACHER_DEVICE)
    t.load_state_dict({k: v.to(TEACHER_DEVICE) for k, v in sd.items()}, strict=True)
    t.half()
    for p in t.parameters():
        p.requires_grad_(False)
    t.eval()
    return t

def soft_targets_for_ds(t_probs, out_shapes):
    outs = []
    for shp in out_shapes:
        if tuple(t_probs.shape[2:]) == tuple(shp):
            outs.append(t_probs)
        else:
            outs.append(_F.interpolate(t_probs, size=shp, mode='area').clamp_min(1e-6))
    return outs

def soft_ce(s_logits, t_probs):
    logp = _F.log_softmax(s_logits.float(), dim=1)
    return -(t_probs * logp).sum(dim=1).mean()

def soft_dice(s_logits, t_probs, eps=1e-6):
    s = _F.softmax(s_logits.float(), dim=1)
    dims = (0, 2, 3, 4)
    inter = (s * t_probs).sum(dims)
    denom = s.sum(dims) + t_probs.sum(dims)
    return (1.0 - (2.0 * inter + eps) / (denom + eps)).mean()

class _SessionOver(Exception):
    pass

class nnUNetTrainerIterKD(nnUNetTrainer):
    """Recovery KD: SGD 1e-4 mom 0.9 nesterov wd 3e-5, PolyLR(power=1)."""
    def __init__(self, plans, configuration, fold, dataset_json,
                 device=torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = RECOVERY_EPOCHS
        self.num_val_iterations_per_epoch = 10
        self.save_every = 5

    def configure_optimizers(self, lr_init=None, new_weight_decay=None):
        optimizer = torch.optim.SGD(self.network.parameters(), RECOVERY_LR,
                                    weight_decay=RECOVERY_WD,
                                    momentum=RECOVERY_MOMENTUM, nesterov=True)
        scheduler = torch.optim.lr_scheduler.PolynomialLR(
            optimizer, total_iters=self.num_epochs, power=1.0)
        self.optimizer, self.lr_scheduler = optimizer, scheduler
        return optimizer, scheduler

    def on_epoch_end(self):
        super().on_epoch_end()
        if self.local_rank == 0 and (self.current_epoch + 1) % 10 == 0:
            try:
                os.makedirs(f'{RES_DIR_GLOBAL}/step_ckpt', exist_ok=True)
                shutil.copy(join(self.output_folder, 'checkpoint_latest.pth'),
                            f'{RES_DIR_GLOBAL}/step_ckpt/latest.pth')
            except Exception as e:
                print('[ckpt-mirror]', e)
        if (time.time() - KERNEL_T0) > SESSION_BUDGET_H * 3600:
            print(f'[budget] {SESSION_BUDGET_H} ч — чистый стоп на эпохе {self.current_epoch}')
            self.save_checkpoint(join(self.output_folder, 'checkpoint_latest.pth'))
            raise _SessionOver

    def train_step(self, batch: dict) -> dict:
        data = batch['data']
        target = batch['target']
        data = data.to(self.device, non_blocking=True)
        if isinstance(target, list):
            target = [t.to(self.device, non_blocking=True) for t in target]
        else:
            target = target.to(self.device, non_blocking=True)
        self.optimizer.zero_grad(set_to_none=True)
        with torch.autocast(self.device.type, enabled=self.device.type == 'cuda'):
            output = self.network(data)
            shapes = [tuple(o.shape[2:]) for o in output]
            with torch.no_grad():
                tp = teacher_probs(data)
            t_list = soft_targets_for_ds(tp, shapes)
            n = len(output)
            ws = [1.0 / (2 ** i) for i in range(n)]
            ws = [w / sum(ws) for w in ws]   # ВАЖНО: sum(ws), не sum(w) — грабля v7
            l = sum(w * (soft_ce(o, t) + soft_dice(o, t))
                    for w, o, t in zip(ws, output, t_list))
        if self.grad_scaler is not None:
            self.grad_scaler.scale(l).backward()
            self.grad_scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.grad_scaler.step(self.optimizer)
            self.grad_scaler.update()
        else:
            l.backward()
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.optimizer.step()
        return {'loss': l.detach().cpu().numpy()}

# --- учителя ---
teachers = [_load_teacher(ckpt['network_weights'])] if TEACHER_FOLD0 else []
if TEACHER_ENSEMBLE:
    ens = sorted(glob.glob('/kaggle/input/**/datasets/**/'
                           'nnUNetTrainerNoMirroring__nnUNetResEncUNetMPlans__3d_fullres/'
                           'fold_*/checkpoint_final.pth', recursive=True))
    by_tr = {}
    for h in ens:
        by_tr.setdefault(os.path.dirname(h), []).append(h)
    full = [v for v in by_tr.values() if len(v) >= 5]
    if full:
        for h in sorted(full[0])[:5]:
            sd_i = torch.load(h, map_location='cpu', weights_only=False)['network_weights']
            teachers.append(_load_teacher(sd_i))
print(f'учитель: {len(teachers)} моделей на {TEACHER_DEVICE}')

@torch.no_grad()
def teacher_probs(data):
    d = data.to(TEACHER_DEVICE, non_blocking=True).half()
    ps = None
    for t in teachers:
        logits = t(d)
        if isinstance(logits, (list, tuple)):
            logits = logits[0]
        p = torch.softmax(logits.float(), dim=1)
        ps = p if ps is None else ps + p
    return (ps / len(teachers)).detach().to(data.device)

# --- итеративные шаги ---
BASE_CH = {'stem': 32, 'enc0': 32, 'enc1': 64, 'enc2': 128, 'enc3': 256, 'enc4': 320, 'enc5': 320}
plan_steps = list(PRUNE_PLAN.items())
current_feats = plans6_to_feats7(FEATS_BASE)      # старт: плотная модель
sd_work = ckpt['network_weights']                  # веса для нарезки (после шага k — recovered)
iter_log = []
PLANS_ORDER = ['stem', 'enc0', 'enc1', 'enc2', 'enc3', 'enc4', 'enc5']

for step_i, (stage, target_ch) in enumerate(plan_steps, 1):
    if (time.time() - KERNEL_T0) > SESSION_BUDGET_H * 3600 * 0.85:
        print('[budget] остановка до шага', step_i, '- продолжите с checkpoint_latest')
        break
    # enc4=enc5 синхронизируем (связанная пара машины резки)
    ratios = {stage: 1 - target_ch / BASE_CH[stage]}
    if stage in ('enc4', 'enc5'):
        other = 'enc5' if stage == 'enc4' else 'enc4'
        ratios[other] = 1 - target_ch / BASE_CH[other]

    print(f'\n===== ШАГ {step_i}/{len(plan_steps)}: {stage} → {target_ch} каналов =====')
    # one-shot резка от текущих (recovered) весов
    sync_plans(current_feats)
    net_c = build_net(feat_override=current_feats)
    net_c.load_state_dict({k: v.cpu() for k, v in sd_work.items()}, strict=True)
    apply_stage_ratios(net_c, ratios)
    feats_after = encoder_feats_from_net(net_c)
    sync_plans(feats_after)
    sd_cut = {k: v.cpu() for k, v in net_c.state_dict().items()}
    n_par = sum(p.numel() for p in net_c.parameters())
    print(f'feats7: {feats_after} | параметры: {n_par/1e6:.1f}M')
    torch.save({'network_weights': sd_cut, 'feats7': feats_after},
               f'{RES_DIR_GLOBAL}/cut_step{step_i}.pth')

    if not RUN_ITERATIVE:
        sd_work, current_feats = sd_cut, feats_after
        d_oneshot = validate_many([(f'STEP{step_i}_ONESHOT', stub_for(sd_cut, net_c))]).get(f'STEP{step_i}_ONESHOT')
        iter_log.append({'step': step_i, 'stage': stage, 'oneshot': d_oneshot, 'recovered': None})
        net_c = net_c.cpu(); del net_c; gc.collect(); torch.cuda.empty_cache()
        continue

    d_oneshot = validate_many([(f'STEP{step_i}_ONESHOT', stub_for(sd_cut, net_c))]).get(f'STEP{step_i}_ONESHOT')
    net_c = net_c.cpu(); del net_c
    gc.collect(); torch.cuda.empty_cache()
    print(f'one-shot: {d_oneshot:.4f}')

    # recovery KD
    plans_rec = load_json(join(ds_work, f'{PLANS_NAME}.json'))
    if 'continue_training' not in plans_rec:
        plans_rec['continue_training'] = False
    trainer = nnUNetTrainerIterKD(plans=plans_rec, configuration=CONFIG, fold=FOLD,
                                  dataset_json=load_json(join(ds_work, 'dataset.json')))
    trainer.num_processes = 2
    trainer.initialize()
    net = trainer.network
    if hasattr(net, '_orig_mod'):
        net = net._orig_mod
    net.load_state_dict({k: v.cuda() for k, v in sd_cut.items()}, strict=True)
    print('cut-веса загружены (strict) | lr =', trainer.optimizer.param_groups[0]['lr'])
    try:
        trainer.run_training()
    except _SessionOver:
        print('[run] бюджет-стоп: чекпоинт сохранён')

    rec_ck = join(trainer.output_folder, 'checkpoint_final.pth')
    if not os.path.isfile(rec_ck):
        rec_ck = join(trainer.output_folder, 'checkpoint_latest.pth')
    rec_sd = torch.load(rec_ck, map_location='cpu', weights_only=False)['network_weights']
    net_rec = build_net(feat_override=feats_after)
    d_rec = validate_many([(f'STEP{step_i}_RECOVERED', stub_for(rec_sd, net_rec))]).get(f'STEP{step_i}_RECOVERED')
    net_rec = net_rec.cpu(); del net_rec
    gc.collect(); torch.cuda.empty_cache()
    print(f'recovered: {d_rec:.4f} (возврат +{d_rec - d_oneshot:.4f})')
    print(f'гейт {DICE_GATE}: {"ПРОЙДЕН" if d_rec >= DICE_GATE else "ПРОВАЛЕН — пайплайн остановлен"}')
    iter_log.append({'step': step_i, 'stage': stage, 'target_ch': target_ch,
                     'oneshot': d_oneshot, 'recovered': d_rec,
                     'params_M': n_par / 1e6, 'feats7': feats_after})
    json.dump(iter_log, open(f'{RES_DIR_GLOBAL}/iterative_log.json', 'w'), indent=1)
    if d_rec < DICE_GATE:
        sd_work, current_feats = None, None
        break
    sd_work, current_feats = rec_sd, feats_after
    # чекпоинт шага для resume между сессиями
    os.makedirs(f'{RES_DIR_GLOBAL}/recovered', exist_ok=True)
    shutil.copy(rec_ck, f'{RES_DIR_GLOBAL}/recovered/after_step{step_i}.pth')

if sd_work is not None:
    FINAL_FEATS = current_feats
    FINAL_SD = sd_work
    json.dump({'feats7': FINAL_FEATS, 'iter_log': iter_log},
              open(f'{RES_DIR_GLOBAL}/final_state.json', 'w'), indent=1)
    print(f'\nИТОГ ФАЗЫ B: feats7={FINAL_FEATS}, лог шагов в iterative_log.json')


In [ ]:
# ============ ФИНАЛЬНАЯ ВАЛИДАЦИЯ PRUNED-МОДЕЛИ (итог фазы B) ============
if sd_work is not None:
    sync_plans(FINAL_FEATS)
    net_fin = build_net(feat_override=FINAL_FEATS)
    net_fin.load_state_dict({k: v.cpu() for k, v in FINAL_SD.items()}, strict=True)
    d_fin = validate_many([('FINAL_PRUNED', stub_for(FINAL_SD, net_fin))]).get('FINAL_PRUNED')
    s_fin = json.load(open(f'{RES_DIR_GLOBAL}/val_cache_FINAL_PRUNED.json')).get('summary')
    print(f'\n===== ФИНАЛЬНАЯ МОДЕЛЬ (после всех шагов) =====')
    print(f'Dice = {d_fin:.4f} | feats7 = {FINAL_FEATS}')
    if s_fin:
        # малые структуры
        import numpy as _np
        mpc = s_fin.get('metric_per_case') or []
        if mpc:
            dj_labels = dataset_json.get('labels') or {}
            small_names = ['Lens_R','Lens_L','Cochlea_R','Cochlea_L','TMJoint_R','TMJoint_L',
                           'Eye_R','Eye_L','OpticNerve_R','OpticNerve_L']
            small_ids = [int(v) for k, v in dj_labels.items() if k in small_names]
            agg = {}
            for organ in mpc[0]['metrics'].keys():
                vals = [c['metrics'][organ].get('Dice') for c in mpc
                        if isinstance(c['metrics'].get(organ), dict) and c['metrics'][organ].get('Dice') is not None]
                if vals:
                    agg[int(organ)] = _np.mean(vals)
            print('малые структуры:')
            for cid in small_ids:
                nm = [k for k, v in dj_labels.items() if int(v) == cid]
                print(f'  {nm[0] if nm else cid:16s} {agg.get(cid, float("nan")):.4f}')
    net_fin = net_fin.cpu(); del net_fin
    gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ============ RUNNER: NNCF-квантизация в subprocess (свои параметры) ============
# Случайные позиции по объёму (НЕ fg-центр) + периферия = прод-подобное покрытие
# (грабля-кризис 0.0004: fg-центрированные train-патчи давали мёртвый INT8).
runner_src = r"""
import sys, json, glob, os, time
import numpy as np
import torch
import openvino as ov
import blosc2
import random

sys.stdout.reconfigure(line_buffering=True)

RES_DIR = sys.argv[1]
DATASET_NAME = sys.argv[2]
PLANS_NAME = sys.argv[3]
DATA_ID = sys.argv[4]
IR_XML = sys.argv[5]
INT8_XML = sys.argv[6]
N_PATCHES = int(sys.argv[7])
N_CASES = int(sys.argv[8])
CT_NORM = bool(int(sys.argv[9]))
PRESET = sys.argv[10]

from batchgenerators.utilities.file_and_folder_operations import join, load_json
plans = load_json(join('/kaggle/working/nnUNet_preprocessed', DATASET_NAME, f'{PLANS_NAME}.json'))
fp = plans['foreground_intensity_properties_per_channel']['0']
print(f'CTNormalization on={CT_NORM}: clip [{fp["percentile_00_5"]}, {fp["percentile_99_5"]}], '
      f'mean={fp["mean"]:.1f} std={fp["std"]:.1f}', flush=True)

def ct_normalize(arr):
    if not CT_NORM:
        return arr.astype('float32')
    a = np.clip(arr, fp['percentile_00_5'], fp['percentile_99_5'])
    return ((a - fp['mean']) / fp['std']).astype('float32')

random.seed(42)
np.random.seed(42)
ps = [56, 192, 192]
data_dir = join('/kaggle/working/nnUNet_preprocessed', DATASET_NAME, DATA_ID)
splits = json.load(open(join('/kaggle/working/nnUNet_preprocessed', DATASET_NAME, 'splits_final.json')))
train_cases = splits[0]['train']
picks = random.sample(train_cases, min(N_CASES, len(train_cases)))
per_case = max(1, N_PATCHES // len(picks))
calib = []
for case in picks:
    img = blosc2.open(join(data_dir, f'{case}.b2nd'))[:].astype('float32')
    img = ct_normalize(img)
    for _ in range(per_case):
        offs = [np.random.randint(0, max(1, img.shape[1 + i] - ps[i] + 1)) for i in range(3)]
        patch = img[:, offs[0]:offs[0]+ps[0], offs[1]:offs[1]+ps[1], offs[2]:offs[2]+ps[2]]
        if patch.shape[1:] != tuple(ps):
            pad = [(0, 0)] + [(0, ps[i] - patch.shape[1 + i]) for i in range(3)]
            patch = np.pad(patch, pad)
        calib.append(torch.from_numpy(np.ascontiguousarray(patch))[None])   # 5D!
print(f'калибровочных патчей: {len(calib)} (случайные позиции)', flush=True)
assert calib

import nncf
ov_core = ov.Core()
ov_model = ov_core.read_model(IR_XML)
calib_dataset = nncf.Dataset(calib, lambda x: {'x': x})
t0 = time.time()
preset = nncf.QuantizationPreset.MIXED if PRESET == 'MIXED' else nncf.QuantizationPreset.PERFORMANCE
q_model = nncf.quantize(ov_model, calib_dataset, preset=preset,
                        subset_size=min(N_PATCHES, len(calib)))
print(f'NNCF done in {(time.time()-t0)/60:.1f} мин', flush=True)
ov.save_model(q_model, INT8_XML)
print('INT8_IR_OK', round(os.path.getsize(INT8_XML[:-4] + '.bin') / 1e6), 'МБ', flush=True)
"""
with open('/kaggle/working/runner_quant.py', 'w', encoding='utf-8') as f:
    f.write(runner_src)
print('runner_quant.py записан (параметризованный)')


In [ ]:
# ============ ФАЗА C1: ONNX → IR FP32 ============
if RUN_EXPORT_INT8 and sd_work is not None:
    import openvino as ov
    ov_core = ov.Core()
    ps = cfg['patch_size']
    net_f_cpu = build_net(feat_override=FINAL_FEATS)
    net_f_cpu.load_state_dict({k: v.cpu() for k, v in FINAL_SD.items()}, strict=True)
    net_f_cpu.eval()
    dummy = torch.randn(1, IN_CH, *ps)      # CPU-экспорт (cuda-экспорт = device-mismatch)
    onnx_path = f'{RES_DIR_GLOBAL}/final_pruned.onnx'
    try:
        torch.onnx.export(net_f_cpu, (dummy,), onnx_path, opset_version=13,
                          input_names=['x'], output_names=['y'],
                          do_constant_folding=True, external_data=False)
    except TypeError:
        torch.onnx.export(net_f_cpu, (dummy,), onnx_path, opset_version=13,
                          input_names=['x'], output_names=['y'],
                          do_constant_folding=True)
    assert not os.path.exists(onnx_path + '.data'), 'external data бага: веса ушли в .onnx.data'
    print('ONNX:', round(os.path.getsize(onnx_path) / 1e6), 'МБ')
    ir_xml = f'{RES_DIR_GLOBAL}/final_pruned_fp32.xml'
    ov.save_model(ov_core.read_model(onnx_path), ir_xml, compress_to_fp16=False)
    print('IR FP32:', round(os.path.getsize(ir_xml[:-4] + '.bin') / 1e6), 'МБ')
    net_f_cpu = net_f_cpu.cpu(); del net_f_cpu
    gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ============ ФАЗА C2: INT8 NNCF в SUBPROCESS (продовая калибровка!) ============
# КРИТИЧНО (кризис 0.0004, cookbook §3): калибровка — CALIB_PATCHES с ПРОД-подобным покрытием
# (случайные позиции + периферия + CTNormalization), НЕ fg-центрированные train-патчи.
if RUN_EXPORT_INT8 and sd_work is not None:
    import subprocess, threading
    INT8_XML = f'{RES_DIR_GLOBAL}/final_pruned_int8.xml'
    cmd = [sys.executable, '/kaggle/working/runner_quant.py',
           RES_DIR_GLOBAL, DATASET_NAME, PLANS_NAME, data_id, ir_xml, INT8_XML,
           str(CALIB_PATCHES), str(CALIB_CASES), str(int(CALIB_CT_NORM)), INT8_PRESET]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    def _stream():
        for line in proc.stdout:
            print('[runner]', line.rstrip(), flush=True)
    th = threading.Thread(target=_stream, daemon=True)
    th.start()
    proc.wait()
    assert proc.returncode == 0 and os.path.exists(INT8_XML), \
        f'runner упал: rc={proc.returncode} (см. [runner]-строки выше)'
    print('INT8 IR:', round(os.path.getsize(INT8_XML[:-4] + '.bin') / 1e6), 'МБ')


In [ ]:
# ============ ARGMAX-ГЕЙТ INT8 vs FP32 (свои 60 патчей: calib жил в runner-процессе) ============
import blosc2
import numpy as np
fp_props = plans_base['foreground_intensity_properties_per_channel']['0']
CT_MEAN, CT_STD = fp_props['mean'], fp_props['std']
CT_P005, CT_P995 = fp_props['percentile_00_5'], fp_props['percentile_99_5']
def _ctn(arr):
    a = np.clip(arr, CT_P005, CT_P995)
    return ((a - CT_MEAN) / CT_STD).astype(np.float32)
data_dir_g = join(ds_work, data_id)
splits_g = json.load(open(join(ds_work, 'splits_final.json')))
train_g = splits_g[FOLD]['train']
random.seed(SEED + 9)
picks_g = random.sample(train_g, 3)
calib = []
for case in picks_g:
    img = blosc2.open(join(data_dir_g, f'{case}.b2nd'))[:].astype('float32')
    img = _ctn(img)
    tgt = blosc2.open(join(data_dir_g, f'{case}_seg.b2nd'))[:].astype('int64')
    fg = (tgt[0] > 0)
    if not fg.any():
        continue
    fgi = np.argwhere(fg)
    for _ in range(20):
        pick = fgi[np.random.randint(0, len(fgi))]
        offs = [max(0, min(int(pick[i]) - ps[i] // 2, img.shape[1+i] - ps[i])) for i in range(3)]
        patch = img[:, offs[0]:offs[0]+ps[0], offs[1]:offs[1]+ps[1], offs[2]:offs[2]+ps[2]]
        if patch.shape[1:] != tuple(ps):
            pad = [(0, 0)] + [(0, ps[i] - patch.shape[1+i]) for i in range(3)]
            patch = np.pad(patch, pad)
        calib.append(torch.from_numpy(np.ascontiguousarray(patch))[None])
print('гейт-патчей:', len(calib))
comp_fp = ov_core.compile_model(ov_core.read_model(ir_xml), 'CPU',
                                {'PERFORMANCE_HINT': 'LATENCY', 'INFERENCE_NUM_THREADS': 4})
int8_xml = INT8_XML
comp_i8 = ov_core.compile_model(ov_core.read_model(int8_xml), 'CPU',
                                {'PERFORMANCE_HINT': 'LATENCY', 'INFERENCE_NUM_THREADS': 4})
agree, total_n = 0, 0
with torch.no_grad():
    for t in calib:
        x = {'x': t.numpy()}
        a = comp_fp(x)['y'].argmax(axis=1)
        b = comp_i8(x)['y'].argmax(axis=1)
        agree += int((a == b).sum())
        total_n += a.size
gate = agree / total_n
print(f'argmax-гейт: {gate:.4f} (n={total_n})')
print('ГЕЙТ ПРОЙДЕН' if gate >= 0.999 else 'ГЕЙТ НИЖЕ 0.999 — пересобрать INT8 (accuracy-aware)')


In [ ]:
# ============ PER-CASE ТАЙМИНГИ: final-FP32 vs final-INT8 (T4 GPU для torch недоступен тут — CPU-бенчмарк IR) ============
# Замер на 4 vCPU CPU (как quantization_final): per-patch latency IR FP32 vs INT8.
# (Финальная правда — локальный бенчмарк на i5-11300H по BENCHMARKS.md, после скачивания весов.)
def bench_ir(xml, n_iter=30):
    comp = ov_core.compile_model(ov_core.read_model(xml), 'CPU',
                                 {'PERFORMANCE_HINT': 'LATENCY', 'INFERENCE_NUM_THREADS': 4})
    x = np.random.randn(1, IN_CH, *ps).astype(np.float32)
    for _ in range(3):
        comp({'x': x})
    ts = []
    for _ in range(n_iter):
        t0 = time.time()
        comp({'x': x})
        ts.append(time.time() - t0)
    ts.sort()
    return ts[len(ts)//2] * 1000

t_fp32 = bench_ir(ir_xml)
t_int8 = bench_ir(INT8_XML)
print(f'FP32: {t_fp32:.0f} мс/патч | INT8: {t_int8:.0f} мс/патч | speedup {t_fp32/t_int8:.2f}x')
json.dump({'ms_fp32': t_fp32, 'ms_int8': t_int8,
           'speedup': t_fp32 / t_int8, 'argmax_gate': gate,
           'feats7': FEATS_FINAL, 'calib_patches': len(calib)},
          open(f'{RES_DIR}/deploy_report.json', 'w'), indent=1)
print('сохранено: deploy_report.json')
print(f'Полное время сессии: {(time.time() - KERNEL_T0)/3600:.2f} ч')


In [ ]:
# ============ ИТОГОВЫЙ ОТЧЁТ ПАЙПЛАЙНА ============
import numpy as np
report = {
    'pipeline': 'first-N pruning final (scan → iterative+recovery → INT8)',
    'prune_plan': PRUNE_PLAN,
    'iterative_log': iter_log if 'iter_log' in globals() else None,
    'final_feats7': FINAL_FEATS if 'FINAL_FEATS' in globals() else None,
    'depth_scan': depth_scan if 'depth_scan' in globals() and depth_scan else None,
    'final_dice_foldval': d_fin if 'd_fin' in globals() else None,
    'int8': (json.load(open(f'{RES_DIR_GLOBAL}/deploy_report.json'))
             if os.path.exists(f'{RES_DIR_GLOBAL}/deploy_report.json') else None),
    'session_h': (time.time() - KERNEL_T0) / 3600,
}
json.dump(report, open(f'{RES_DIR_GLOBAL}/PIPELINE_REPORT.json', 'w'), indent=1)
print(json.dumps({k: v for k, v in report.items() if k != 'depth_scan'},
                 indent=1, ensure_ascii=False, default=str)[:2000])
print(f'\nПолное время сессии: {report["session_h"]:.2f} ч')
print('Все артефакты в:', RES_DIR_GLOBAL)
